# Create Kauffman Foundation Awards (GRANT PATTERN, WP REST org-level)

Ewing Marion Kauffman Foundation (Kansas City, founded 1966) funds
organizations — universities, nonprofits, school districts — directly,
not individual researchers. Its grant database is published on the
foundation's own site via the WordPress REST API's `grant` custom
post type, with rich per-grant metadata (USD amount, approval date,
grantee city, grantee web URL) plus three taxonomies
(states / strategies / grant-types).

**Awarding body:** Ewing Marion Kauffman Foundation - F4320306140 (US, DOI 10.13039/100000865)

**Source:** https://www.kauffman.org/wp-json/wp/v2/grant?per_page=100 (337 grants), plus the three taxonomy endpoints (`/states`, `/strategies`, `/grant-types`) to resolve term IDs to names.

**Schema choices:**
- One row per grant. `funder_award_id` = `kauffman-{wp_post_id}` (stable WP post ID).
- `funder_scheme` = primary `strategy` taxonomy term (Past Strategy / Entrepreneurship / College Access and Completion / Impact Other / Real World Learning / +1).
- `funding_type` derived via CASE on `grant_type`: `research` when grant_type is `Research Grant` (22 rows in current corpus), else `grant`.
- `amount` and `currency` populated from `meta.kauffman_grant_approved_amount` (USD, 100% coverage).
- `start_year` from `meta.kauffman_grant_approved_on` (100% coverage, year range 1997-2026).
- `lead_investigator` shaped as ORG-only — Kauffman funds orgs, not PIs: `given_name`/`family_name`/`orcid` NULL, `affiliation.name` = grantee organization, `affiliation.country` = `US` (Kauffman's `states` taxonomy is exclusively US state codes). Same model as Hewlett Foundation (priority 86).
- `landing_page_url` = WP post link (preserves the canonical foundation-side page); the grantee organization's own URL ships separately as metadata in the description text.
- Multi-valued taxonomies: first value canonical, full lists kept in dedicated columns (`strategies_all`, `grant_types_all`) for downstream filtering.

**S3 location:** `s3a://openalex-ingest/awards/kauffman/kauffman_grants.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kauffman_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/kauffman/kauffman_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.kauffman_raw;

## Step 1.5: Inspect raw + per-strategy/year distribution

Expected: ~337 grants, 100% on grantee_org + description + amount + start_year + strategy + grant_type + grantee_city, ~99% on grantee_state, ~88% on grantee_url. Year range 1997-2026. Strategy split dominated by Past Strategy + Entrepreneurship + College Access and Completion.

In [ ]:
%sql
DESCRIBE openalex.awards.kauffman_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.kauffman_raw LIMIT 5;

In [ ]:
%sql
-- Strategy distribution.
SELECT strategy, COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       ROUND(AVG(TRY_CAST(amount AS DOUBLE)), 0) AS avg_amount
FROM openalex.awards.kauffman_raw
GROUP BY strategy
ORDER BY rows DESC;

In [ ]:
%sql
-- Grant_type distribution. Research Grant rows will route to funding_type='research'; others to 'grant'.
SELECT grant_type, COUNT(*) AS rows
FROM openalex.awards.kauffman_raw
GROUP BY grant_type
ORDER BY rows DESC;

In [ ]:
%sql
-- Year range.
SELECT MIN(start_year) AS min_year, MAX(start_year) AS max_year, COUNT(DISTINCT start_year) AS years
FROM openalex.awards.kauffman_raw
WHERE start_year IS NOT NULL;

## Step 1.6: Fail-fast - verify Kauffman Foundation funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306140;

## Step 2: Transform to award schema

Per-row `display_name` = `Kauffman {grant_type} - {grantee_org} ({start_year})`. `description` is the source's grant-purpose blurb. `lead_investigator` shaped as org-only (given/family/orcid NULL, affiliation.name = grantee_org).

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kauffman_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306140  -- Ewing Marion Kauffman Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT('Kauffman ', COALESCE(n.grant_type, 'Grant'),
           ' - ', n.grantee_org,
           CASE WHEN n.start_year IS NOT NULL THEN CONCAT(' (', n.start_year, ')') ELSE '' END) AS display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    CASE
        WHEN LOWER(n.grant_type) RLIKE 'research' THEN 'research'
        ELSE 'grant'
    END AS funding_type,
    n.strategy AS funder_scheme,
    'kauffman_foundation' AS provenance,
    TRY_TO_DATE(n.approved_on, 'yyyy-MM-dd') AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.start_year AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN n.grantee_org IS NULL OR n.grantee_org = '' THEN NULL
        ELSE struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.approved_on, 'yyyy-MM-dd') AS role_start,
            struct(
                n.grantee_org AS name,
                CAST('US' AS STRING) AS country,  -- Kauffman states taxonomy is US-only
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.kauffman_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.grantee_org IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 139

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'kauffman_foundation' AND priority = 139;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    139 AS priority  -- Ewing Marion Kauffman Foundation
FROM openalex.awards.kauffman_awards;

## Step 6: Verification

Expect ~337 rows, 100% amount (USD-only), 100% on grantee_org + description + start_year + funder_scheme. funding_type=research for the 22 Research Grant rows; funding_type=grant for the rest.

In [ ]:
%sql
SELECT COUNT(*) AS total_kauffman_rows FROM openalex.awards.kauffman_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.kauffman_awards;

In [ ]:
%sql
-- §6.3 Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_grantee,
    COUNT(start_year) AS has_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.kauffman_awards;

In [ ]:
%sql
-- §6.7 amount sanity. Expect 100% populated, all USD, $25K min, $168M max.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    MIN(amount) AS min_amount, MAX(amount) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.kauffman_awards;

In [ ]:
%sql
-- funding_type CASE result.
SELECT funding_type, COUNT(*) AS rows
FROM openalex.awards.kauffman_awards
GROUP BY funding_type
ORDER BY rows DESC;

In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) AS rows, ROUND(SUM(amount), 0) AS yearly_total_usd
FROM openalex.awards.kauffman_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year;

In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.kauffman_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample 10 rows for eyeball QA.
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       funder_scheme AS strategy, funding_type, start_year, amount,
       SUBSTRING(lead_investigator.affiliation.name, 1, 45) AS grantee_org
FROM openalex.awards.kauffman_awards
ORDER BY start_year DESC, amount DESC
LIMIT 10;